# CS6493 Topic 4 — Generation Module (Self-Contained Google Colab Notebook)

This notebook is designed for the **generation** part of your NLP final project (**Topic 4: Retrieval-Augmented Generation for Knowledge-Intensive Tasks**).

It is designed to be as **self-contained as possible for Google Colab**:

- runs in **Colab** with local model download
- compares **zero-shot/base** vs **instruction-tuned** generation models
- defaults to **Qwen2.5-7B** vs **Qwen2.5-7B-Instruct**
- uses **4-bit quantization** to fit Colab GPU memory more easily
- loads data directly from Hugging Face datasets for a **smoke test / oracle-context mode**
- also supports **team retrieval JSONL mode** for your actual final experiments
- saves raw outputs and summary tables to disk

## Important note
This notebook has two modes:

1. **oracle mode**  
   Uses contexts already present in the dataset (or gold-like evidence) so you can run your **generation module independently** before your team retrieval pipeline is fully integrated.

2. **team JSONL mode**  
   Uses retrieval outputs exported by your teammate.  
   This is the mode you should use for your **actual final report experiments** about retrieval–generation interaction.

So this notebook is self-contained enough to let you run the generation comparison first, but for the final project write-up you should still switch to **team JSONL mode** once retrieval outputs are ready.


## Recommended Colab runtime

In Colab, go to:

**Runtime → Change runtime type → Hardware accelerator → GPU**

A **T4** is usually enough for one **7B model at a time** with **4-bit quantization**.  
This notebook deliberately loads the two models **sequentially** instead of at the same time to reduce OOM risk.


In [ ]:

# Check your runtime first
!nvidia-smi


Tue Mar 31 13:24:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

# Install dependencies
%pip install -q -U transformers accelerate bitsandbytes datasets pandas tqdm sentencepiece evaluate scikit-learn huggingface_hub


## Optional: mount Google Drive

Set `USE_GOOGLE_DRIVE = True` if you want:

- model cache to survive Colab restarts
- outputs saved to your Drive
- retrieval JSONL files loaded from Drive

If you do not need that, leave it as `False`.


In [ ]:

from pathlib import Path
import os

USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/cs6493_topic4_generation")
else:
    BASE_DIR = Path("/content/cs6493_topic4_generation")

BASE_DIR.mkdir(parents=True, exist_ok=True)

HF_HOME = BASE_DIR / "hf_cache"
OUTPUT_DIR = BASE_DIR / "outputs"
INPUT_DIR = BASE_DIR / "inputs"

HF_HOME.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME)
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("BASE_DIR =", BASE_DIR)
print("HF_HOME  =", HF_HOME)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("INPUT_DIR  =", INPUT_DIR)


BASE_DIR = /content/cs6493_topic4_generation
HF_HOME  = /content/cs6493_topic4_generation/hf_cache
OUTPUT_DIR = /content/cs6493_topic4_generation/outputs
INPUT_DIR  = /content/cs6493_topic4_generation/inputs


In [ ]:

import gc
import json
import math
import random
import re
import itertools
import textwrap
import warnings
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


## Configuration

### Model preset
- `qwen_7b`: stronger, better matches the project expectation, but heavier
- `qwen_3b`: fallback if 7B causes OOM in your Colab session

### Input mode
- `oracle`: fully self-contained smoke test using dataset-provided contexts/evidence
- `team_jsonl`: use your teammate's retrieval outputs

### Domains
Default is:

- `hotpotqa`
- `pubmedqa`
- `financebench`

`natural_questions` is supported too, but it is much heavier, so it is **disabled by default**.


In [ ]:

# =========================
# Main configuration
# =========================

SEED = 42
random.seed(SEED)

MODEL_PRESET = "qwen_7b"   # "qwen_7b" or "qwen_3b"

MODEL_OPTIONS = {
    "qwen_7b": {
        "base": "Qwen/Qwen2.5-7B",
        "instruct": "Qwen/Qwen2.5-7B-Instruct",
    },
    "qwen_3b": {
        "base": "Qwen/Qwen2.5-3B",
        "instruct": "Qwen/Qwen2.5-3B-Instruct",
    },
}

BASE_MODEL_NAME = MODEL_OPTIONS[MODEL_PRESET]["base"]
INSTRUCT_MODEL_NAME = MODEL_OPTIONS[MODEL_PRESET]["instruct"]

INPUT_MODE = "oracle"   # "oracle" or "team_jsonl"
TEAM_JSONL_PATH = INPUT_DIR / "team_retrieval.jsonl"

DOMAINS_TO_RUN = ["hotpotqa", "pubmedqa", "financebench"]   # add "natural_questions" only if you want it
TOP_K_CONTEXTS = 3

# Keep these small first, then increase after a smoke test passes
SAMPLES_PER_DOMAIN = {
    "hotpotqa": 12,
    "pubmedqa": 12,
    "financebench": 12,
    "natural_questions": 8,
}

# Natural Questions is large; streaming avoids giant local downloads
USE_STREAMING_FOR_NQ = True

USE_4BIT = True
FORCE_CPU = False

MAX_NEW_TOKENS = 64
TEMPERATURE = 0.0
DO_SAMPLE = False

OUTPUT_PREFIX = f"{MODEL_PRESET}_topic4_generation"

print("BASE_MODEL_NAME     =", BASE_MODEL_NAME)
print("INSTRUCT_MODEL_NAME =", INSTRUCT_MODEL_NAME)
print("INPUT_MODE          =", INPUT_MODE)
print("DOMAINS_TO_RUN      =", DOMAINS_TO_RUN)


BASE_MODEL_NAME     = Qwen/Qwen2.5-7B
INSTRUCT_MODEL_NAME = Qwen/Qwen2.5-7B-Instruct
INPUT_MODE          = oracle
DOMAINS_TO_RUN      = ['hotpotqa', 'pubmedqa', 'financebench']


## Optional: Hugging Face login

Most of the models and datasets used here are public, so you often do **not** need to log in.

Only run this if:
- a model download complains about authentication, or
- you want better Hugging Face rate limits


In [ ]:

# Optional:
# from huggingface_hub import login
# login()


## Utilities


In [ ]:

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def normalize_text(text: Any) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_answer_for_ynm(text: Any) -> str:
    t = normalize_text(text)
    if re.search(r"\byes\b", t):
        return "yes"
    if re.search(r"\bno\b", t):
        return "no"
    if re.search(r"\bmaybe\b", t):
        return "maybe"
    return t

def exact_match_score(pred: Any, gold: Any) -> float:
    pred_n = normalize_answer_for_ynm(pred)
    gold_n = normalize_answer_for_ynm(gold)
    return float(pred_n == gold_n)

def token_f1_score(pred: Any, gold: Any) -> float:
    pred_tokens = normalize_answer_for_ynm(pred).split()
    gold_tokens = normalize_answer_for_ynm(gold).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = {}
    for t in pred_tokens:
        common[t] = min(pred_tokens.count(t), gold_tokens.count(t))
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def safe_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def context_to_text(ctx: Any) -> str:
    if isinstance(ctx, str):
        return ctx.strip()
    if isinstance(ctx, dict):
        title = str(ctx.get("title", "")).strip()
        text = str(ctx.get("text", ctx.get("page_content", ctx.get("evidence_text", "")))).strip()
        if title and text:
            return f"[{title}] {text}"
        return (title or text).strip()
    return str(ctx).strip()

def contexts_to_list(contexts: Any, top_k: int = 3) -> List[str]:
    out = []
    for c in safe_list(contexts):
        s = context_to_text(c)
        if s:
            out.append(s)
    return out[:top_k]

def join_contexts(contexts: List[str]) -> str:
    pieces = []
    for i, ctx in enumerate(contexts, start=1):
        pieces.append(f"[Document {i}] {ctx}")
    return "\n\n".join(pieces)

def is_abstention(text: Any) -> bool:
    t = str(text).lower()
    triggers = [
        "insufficient evidence",
        "not enough information",
        "cannot answer",
        "can't answer",
        "unknown",
        "not provided in the context",
    ]
    return any(x in t for x in triggers)

def prediction_in_context(pred: Any, contexts: Any) -> float:
    pred_n = normalize_text(pred)
    ctx_n = normalize_text(" ".join(contexts_to_list(contexts, top_k=999)))
    if not pred_n:
        return 0.0
    return float(pred_n in ctx_n)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Dataset adapters

This notebook supports the 4 datasets you listed:

1. HotpotQA  
2. Natural Questions  
3. PubMedQA  
4. FinanceBench

In **oracle mode**, each dataset is converted into a unified format:

- `id`
- `domain`
- `question`
- `answer`
- `contexts`

That way, the same generation code can run across all domains.


In [ ]:

DATASET_CONFIGS = {
    "hotpotqa": {
        "path": "hotpotqa/hotpot_qa",
        "name": "distractor",
        "split": "validation",
    },
    "natural_questions": {
        "path": "google-research-datasets/natural_questions",
        "split": "validation",
    },
    "pubmedqa": {
        "path": "qiaojin/PubMedQA",
        "name": "pqa_labeled",
        "split": "train",
    },
    "financebench": {
        "path": "PatronusAI/financebench",
        "split": "train",
    },
}


In [ ]:

def hotpotqa_to_record(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    titles = item.get("context", {}).get("title", [])
    sent_lists = item.get("context", {}).get("sentences", [])
    support_titles = set(item.get("supporting_facts", {}).get("title", []))

    contexts = []
    for title, sentences in zip(titles, sent_lists):
        text = " ".join(sentences).strip()
        contexts.append({
            "title": title,
            "text": text,
            "is_supporting_title": title in support_titles,
        })

    contexts = sorted(contexts, key=lambda x: (not x["is_supporting_title"], x["title"]))
    contexts = contexts_to_list(contexts, top_k=TOP_K_CONTEXTS)

    if not item.get("question") or not item.get("answer"):
        return None

    return {
        "id": item.get("id", ""),
        "domain": "hotpotqa",
        "question": item["question"],
        "answer": item["answer"],
        "contexts": contexts,
        "retrieval_method": "oracle_context",
        "top_k": TOP_K_CONTEXTS,
    }

def pubmedqa_to_record(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    context_parts = item.get("context", [])
    contexts = []
    if isinstance(context_parts, list):
        contexts = [" ".join([str(x) for x in context_parts if str(x).strip()])]
    elif isinstance(context_parts, str):
        contexts = [context_parts]

    question = item.get("question", "")
    answer = item.get("final_decision", "")

    if not question or not answer:
        return None

    return {
        "id": str(item.get("pubid", "")),
        "domain": "pubmedqa",
        "question": question,
        "answer": answer,
        "contexts": contexts_to_list(contexts, top_k=TOP_K_CONTEXTS),
        "retrieval_method": "oracle_context",
        "top_k": TOP_K_CONTEXTS,
    }

def financebench_to_record(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    evidence = item.get("evidence", [])
    contexts = []
    for ev in safe_list(evidence):
        if isinstance(ev, dict):
            txt = ev.get("evidence_text", "")
            if txt:
                contexts.append({
                    "title": item.get("doc_name", ""),
                    "text": txt,
                })
        elif isinstance(ev, str) and ev.strip():
            contexts.append(ev)

    question = item.get("question", "")
    answer = item.get("answer", "")

    if not question or not answer:
        return None

    return {
        "id": item.get("financebench_id", ""),
        "domain": "financebench",
        "question": question,
        "answer": answer,
        "contexts": contexts_to_list(contexts, top_k=TOP_K_CONTEXTS),
        "retrieval_method": "oracle_evidence",
        "top_k": TOP_K_CONTEXTS,
    }

def _nq_question_text(item: Dict[str, Any]) -> str:
    q = item.get("question", "")
    if isinstance(q, dict):
        return str(q.get("text", "")).strip()
    return str(q).strip()

def _nq_doc_tokens(item: Dict[str, Any]):
    doc = item.get("document", {})
    toks = doc.get("tokens", {})
    if isinstance(toks, dict):
        token_list = toks.get("token", [])
        is_html = toks.get("is_html", [False] * len(token_list))
        return token_list, is_html
    elif isinstance(toks, list):
        token_list = [str(x.get("token", "")) for x in toks]
        is_html = [bool(x.get("is_html", False)) for x in toks]
        return token_list, is_html
    return [], []

def _nq_slice_tokens(tokens: List[str], is_html: List[bool], start_token: int, end_token: int) -> str:
    start_token = max(0, int(start_token))
    end_token = max(start_token, int(end_token))
    pieces = []
    for tok, html_flag in zip(tokens[start_token:end_token], is_html[start_token:end_token]):
        if not html_flag:
            pieces.append(str(tok))
    text = " ".join(pieces)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def natural_questions_to_record(item: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    question = _nq_question_text(item)
    tokens, is_html = _nq_doc_tokens(item)
    annotations = item.get("annotations", [])

    if not question or not tokens or not annotations:
        return None

    ann = annotations[0]

    answer = ""
    short_answers = ann.get("short_answers", [])
    if short_answers:
        sa = short_answers[0]
        answer = _nq_slice_tokens(tokens, is_html, sa.get("start_token", 0), sa.get("end_token", 0))

    if not answer:
        yes_no = str(ann.get("yes_no_answer", "")).strip().lower()
        if yes_no and yes_no not in {"none", "null"}:
            answer = yes_no

    long_answer = ann.get("long_answer", {})
    context_text = ""
    if isinstance(long_answer, dict):
        context_text = _nq_slice_tokens(
            tokens,
            is_html,
            long_answer.get("start_token", 0),
            long_answer.get("end_token", 0),
        )

    # If no short answer but long answer exists, use long answer as fallback label
    if not answer and context_text:
        answer = context_text

    if not context_text:
        # fallback: take first non-html chunk
        non_html_tokens = [tok for tok, flag in zip(tokens, is_html) if not flag]
        context_text = " ".join(non_html_tokens[:300]).strip()

    if not question or not answer or not context_text:
        return None

    title = ""
    if isinstance(item.get("document", {}), dict):
        title = str(item.get("document", {}).get("title", "")).strip()

    return {
        "id": str(item.get("id", "")),
        "domain": "natural_questions",
        "question": question,
        "answer": answer,
        "contexts": contexts_to_list([{"title": title, "text": context_text}], top_k=TOP_K_CONTEXTS),
        "retrieval_method": "oracle_long_answer",
        "top_k": TOP_K_CONTEXTS,
    }

ADAPTERS = {
    "hotpotqa": hotpotqa_to_record,
    "pubmedqa": pubmedqa_to_record,
    "financebench": financebench_to_record,
    "natural_questions": natural_questions_to_record,
}


In [ ]:

def load_oracle_dataset(domain: str, sample_size: Optional[int] = None) -> pd.DataFrame:
    cfg = DATASET_CONFIGS[domain]
    adapter = ADAPTERS[domain]
    sample_size = sample_size or SAMPLES_PER_DOMAIN.get(domain, 10)

    kwargs = {
        "path": cfg["path"],
        "split": cfg["split"],
    }
    if "name" in cfg:
        kwargs["name"] = cfg["name"]

    records = []

    # Natural Questions is large; streaming keeps it Colab-friendlier
    if domain == "natural_questions" and USE_STREAMING_FOR_NQ:
        ds = load_dataset(**kwargs, streaming=True)
        for item in ds:
            rec = adapter(item)
            if rec is not None:
                records.append(rec)
            if len(records) >= sample_size:
                break
    else:
        ds = load_dataset(**kwargs)
        sample_size = min(sample_size, len(ds))
        ds = ds.select(range(sample_size))
        for item in ds:
            rec = adapter(item)
            if rec is not None:
                records.append(rec)

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError(f"No valid records loaded for domain={domain}")

    # final cleanup
    df["question"] = df["question"].fillna("").astype(str)
    df["answer"] = df["answer"].fillna("").astype(str)
    df["contexts"] = df["contexts"].apply(lambda x: contexts_to_list(x, top_k=TOP_K_CONTEXTS))
    return df

# Quick smoke test on one domain if you want:
# smoke_df = load_oracle_dataset("hotpotqa", sample_size=3)
# display(smoke_df.head())


## Team retrieval JSONL mode

For your **actual final experiments**, the cleanest handoff from the retrieval teammate is a JSONL file like this:

```json
{
  "id": "sample_001",
  "domain": "hotpotqa",
  "question": "Who is the director of Inception, and what is his nationality?",
  "answer": "Christopher Nolan, British-American",
  "contexts": [
    {"title": "Inception", "text": "Inception is a 2010 science fiction action film written and directed by Christopher Nolan ..."},
    {"title": "Christopher Nolan", "text": "Christopher Edward Nolan is a British-American filmmaker ..."}
  ],
  "retrieval_method": "bm25",
  "top_k": 3
}
```

This notebook accepts:
- `contexts` as a list of `{title, text}` dictionaries
- or `contexts` as a list of plain strings


In [ ]:

def load_retrieval_jsonl(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"JSONL file not found: {path}")

    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append({
                "id": obj.get("id", ""),
                "domain": obj.get("domain", "unknown"),
                "question": obj.get("question", ""),
                "answer": obj.get("answer", ""),
                "contexts": contexts_to_list(obj.get("contexts", []), top_k=obj.get("top_k", TOP_K_CONTEXTS)),
                "retrieval_method": obj.get("retrieval_method", "unknown"),
                "top_k": obj.get("top_k", TOP_K_CONTEXTS),
            })

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError("The JSONL file was read successfully, but no valid records were found.")
    return df


## Build the evaluation dataframe


In [ ]:

if INPUT_MODE == "oracle":
    frames = []
    for domain in DOMAINS_TO_RUN:
        print(f"Loading oracle data for: {domain}")
        frames.append(load_oracle_dataset(domain, sample_size=SAMPLES_PER_DOMAIN.get(domain, 10)))
    eval_df = pd.concat(frames, ignore_index=True)

elif INPUT_MODE == "team_jsonl":
    eval_df = load_retrieval_jsonl(TEAM_JSONL_PATH)

else:
    raise ValueError("INPUT_MODE must be 'oracle' or 'team_jsonl'")

print("Evaluation dataframe shape:", eval_df.shape)
display(eval_df.head(3))


Loading oracle data for: hotpotqa


Loading oracle data for: pubmedqa
Loading oracle data for: financebench
Evaluation dataframe shape: (36, 7)


,id,domain,question,answer,contexts,retrieval_method,top_k
0,5a8b57f25542995d1e6f1371,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"[[Ed Wood] Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director., [Scott Derrickson] Scott Derrickson (born July 16, 1966) ...",oracle_context,3
1,5a8c7595554299585d9e36b6,hotpotqa,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,"[[Kiss and Tell (1945 film)] Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer. In the film, two teenage girls cause their respective parents...",oracle_context,3
2,5a85ea095542994775f606a8,hotpotqa,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,"[[Animorphs] Animorphs is a science fantasy series of young adult books written by Katherine Applegate and her husband Michael Grant, writing together under the name K. A. Applegate, and published...",oracle_context,3


## Prompt design

For a fair comparison:

- the **base model** gets a plain **zero-shot prompt**
- the **instruction-tuned model** gets a structured **instruction/chat prompt**
- the retrieved contexts stay the same

So the main variable is **model type**, not the evidence.


In [ ]:

def build_base_prompt(question: str, contexts: List[str]) -> str:
    context_block = join_contexts(contexts)
    return f"""Answer the question using only the retrieved context.

Retrieved context:
{context_block}

Question: {question}

Rules:
- If the evidence is insufficient, answer exactly: Insufficient evidence.
- Keep the answer concise.
- Do not add unsupported claims.

Answer:"""

def build_instruct_messages(question: str, contexts: List[str]) -> List[Dict[str, str]]:
    context_block = join_contexts(contexts)
    return [
        {
            "role": "system",
            "content": (
                "You are a careful question-answering assistant. "
                "Answer only from the retrieved context. "
                "If the evidence is insufficient, answer exactly: Insufficient evidence. "
                "Keep the answer concise and do not add unsupported claims."
            ),
        },
        {
            "role": "user",
            "content": f"Retrieved context:\n{context_block}\n\nQuestion: {question}",
        },
    ]


## Model loading

This notebook loads the two models **one after another**, not simultaneously:

1. load base model → run all samples → free memory  
2. load instruct model → run all samples → free memory

That is much safer for Colab memory.


In [ ]:

def pick_compute_dtype():
    if not torch.cuda.is_available() or FORCE_CPU:
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    # bf16 is best on newer GPUs; T4 usually uses fp16
    return torch.bfloat16 if major >= 8 else torch.float16

COMPUTE_DTYPE = pick_compute_dtype()
print("COMPUTE_DTYPE =", COMPUTE_DTYPE)

def make_quantization_config():
    if not USE_4BIT or not torch.cuda.is_available() or FORCE_CPU:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

def load_model_and_tokenizer(model_name: str):
    clear_memory()
    quant_config = make_quantization_config()

    model_kwargs = {
        "low_cpu_mem_usage": True,
    }

    if quant_config is not None:
        model_kwargs["quantization_config"] = quant_config
        model_kwargs["device_map"] = "auto"
    else:
        if torch.cuda.is_available() and not FORCE_CPU:
            model_kwargs["torch_dtype"] = COMPUTE_DTYPE
            model_kwargs["device_map"] = "auto"
        else:
            model_kwargs["torch_dtype"] = torch.float32

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    model.eval()

    return tokenizer, model

def get_inference_device():
    if torch.cuda.is_available() and not FORCE_CPU:
        return "cuda"
    return "cpu"


COMPUTE_DTYPE = torch.float16


In [ ]:

@torch.inference_mode()
def generate_one_answer(model, tokenizer, question: str, contexts: List[str], prompt_mode: str) -> str:
    device = get_inference_device()

    if prompt_mode == "base":
        prompt = build_base_prompt(question, contexts)
        inputs = tokenizer(prompt, return_tensors="pt")
    elif prompt_mode == "instruct":
        messages = build_instruct_messages(question, contexts)
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(prompt, return_tensors="pt")
    else:
        raise ValueError("prompt_mode must be 'base' or 'instruct'")

    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    new_tokens = generated[0][input_len:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # keep the output concise
    text = re.sub(r"^answer\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    first_line = text.splitlines()[0].strip() if text.splitlines() else text.strip()
    return first_line

def run_model_on_eval_df(
    eval_df: pd.DataFrame,
    model_name: str,
    model_label: str,
    prompt_mode: str,
) -> pd.DataFrame:
    tokenizer, model = load_model_and_tokenizer(model_name)

    rows = []
    iterator = tqdm(eval_df.to_dict("records"), total=len(eval_df), desc=f"Running {model_label}")

    for row in iterator:
        pred = generate_one_answer(
            model=model,
            tokenizer=tokenizer,
            question=row["question"],
            contexts=row["contexts"],
            prompt_mode=prompt_mode,
        )

        rows.append({
            **row,
            "model_label": model_label,
            "model_name": model_name,
            "prompt_mode": prompt_mode,
            "prediction": pred,
        })

    # free memory before next model
    del model
    del tokenizer
    clear_memory()

    return pd.DataFrame(rows)


## Run the comparison

The next two cells do the actual experiment:

- zero-shot/base model
- instruction-tuned model


In [ ]:

base_results = run_model_on_eval_df(
    eval_df=eval_df,
    model_name=BASE_MODEL_NAME,
    model_label="base_zero_shot",
    prompt_mode="base",
)

display(base_results.head(3))


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Running base_zero_shot:   0%|          | 0/36 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,id,domain,question,answer,contexts,retrieval_method,top_k,model_label,model_name,prompt_mode,prediction
0,5a8b57f25542995d1e6f1371,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"[[Ed Wood] Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director., [Scott Derrickson] Scott Derrickson (born July 16, 1966) ...",oracle_context,3,base_zero_shot,Qwen/Qwen2.5-7B,base,Insufficient evidence.
1,5a8c7595554299585d9e36b6,hotpotqa,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,"[[Kiss and Tell (1945 film)] Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer. In the film, two teenage girls cause their respective parents...",oracle_context,3,base_zero_shot,Qwen/Qwen2.5-7B,base,Insufficient evidence.
2,5a85ea095542994775f606a8,hotpotqa,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,"[[Animorphs] Animorphs is a science fantasy series of young adult books written by Katherine Applegate and her husband Michael Grant, writing together under the name K. A. Applegate, and published...",oracle_context,3,base_zero_shot,Qwen/Qwen2.5-7B,base,"The Hork-Bajir Chronicles is a science fantasy young adult series, told in first person, with a set of companion books narrating the stories of enslaved worlds and alien species."


In [18]:

instruct_results = run_model_on_eval_df(
    eval_df=eval_df,
    model_name=INSTRUCT_MODEL_NAME,
    model_label="instruction_tuned",
    prompt_mode="instruct",
)

display(instruct_results.head(3))


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Running instruction_tuned:   0%|          | 0/36 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,id,domain,question,answer,contexts,retrieval_method,top_k,model_label,model_name,prompt_mode,prediction
0,5a8b57f25542995d1e6f1371,hotpotqa,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"[[Ed Wood] Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director., [Scott Derrickson] Scott Derrickson (born July 16, 1966) ...",oracle_context,3,instruction_tuned,Qwen/Qwen2.5-7B-Instruct,instruct,"Yes, both Scott Derrickson and Ed Wood were American."
1,5a8c7595554299585d9e36b6,hotpotqa,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,"[[Kiss and Tell (1945 film)] Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer. In the film, two teenage girls cause their respective parents...",oracle_context,3,instruction_tuned,Qwen/Qwen2.5-7B-Instruct,instruct,United States ambassador to Ghana and to Czechoslovakia and also served as Chief of Protocol of the United States.
2,5a85ea095542994775f606a8,hotpotqa,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,"[[Animorphs] Animorphs is a science fantasy series of young adult books written by Katherine Applegate and her husband Michael Grant, writing together under the name K. A. Applegate, and published...",oracle_context,3,instruction_tuned,Qwen/Qwen2.5-7B-Instruct,instruct,"Animorphs is a science fantasy young adult series told in first person, with a set of companion books like The Hork-Bajir Chronicles that narrate stories of enslaved worlds and alien species."


## Combine and score results


In [19]:

all_results = pd.concat([base_results, instruct_results], ignore_index=True)

all_results["exact_match"] = all_results.apply(
    lambda row: exact_match_score(row["prediction"], row["answer"]),
    axis=1,
)
all_results["token_f1"] = all_results.apply(
    lambda row: token_f1_score(row["prediction"], row["answer"]),
    axis=1,
)
all_results["abstained"] = all_results["prediction"].apply(is_abstention).astype(float)
all_results["prediction_in_context"] = all_results.apply(
    lambda row: prediction_in_context(row["prediction"], row["contexts"]),
    axis=1,
)

summary_by_domain = (
    all_results
    .groupby(["domain", "model_label"], as_index=False)
    .agg(
        n=("id", "count"),
        em=("exact_match", "mean"),
        f1=("token_f1", "mean"),
        abstain_rate=("abstained", "mean"),
        pred_in_context_rate=("prediction_in_context", "mean"),
    )
    .sort_values(["domain", "model_label"])
)

summary_overall = (
    all_results
    .groupby(["model_label"], as_index=False)
    .agg(
        n=("id", "count"),
        em=("exact_match", "mean"),
        f1=("token_f1", "mean"),
        abstain_rate=("abstained", "mean"),
        pred_in_context_rate=("prediction_in_context", "mean"),
    )
    .sort_values(["model_label"])
)

display(summary_by_domain)
display(summary_overall)


,domain,model_label,n,em,f1,abstain_rate,pred_in_context_rate
0,financebench,base_zero_shot,12,0.000000,0.000000,0.833333,0.000000
1,financebench,instruction_tuned,12,0.083333,0.152273,0.416667,0.083333
2,hotpotqa,base_zero_shot,12,0.166667,0.198994,0.500000,0.166667
3,hotpotqa,instruction_tuned,12,0.416667,0.519365,0.166667,0.250000
4,pubmedqa,base_zero_shot,12,0.000000,0.000000,1.000000,0.000000
5,pubmedqa,instruction_tuned,12,0.000000,0.000000,1.000000,0.000000


,model_label,n,em,f1,abstain_rate,pred_in_context_rate
0,base_zero_shot,36,0.055556,0.066331,0.777778,0.055556
1,instruction_tuned,36,0.166667,0.223880,0.527778,0.111111


## Save outputs


In [20]:

raw_csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_raw_results.csv"
domain_csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_summary_by_domain.csv"
overall_csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_summary_overall.csv"
jsonl_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_raw_results.jsonl"

all_results.to_csv(raw_csv_path, index=False)
summary_by_domain.to_csv(domain_csv_path, index=False)
summary_overall.to_csv(overall_csv_path, index=False)

with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in all_results.to_dict("records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:")
print("-", raw_csv_path)
print("-", domain_csv_path)
print("-", overall_csv_path)
print("-", jsonl_path)


Saved:
- /content/cs6493_topic4_generation/outputs/qwen_7b_topic4_generation_raw_results.csv
- /content/cs6493_topic4_generation/outputs/qwen_7b_topic4_generation_summary_by_domain.csv
- /content/cs6493_topic4_generation/outputs/qwen_7b_topic4_generation_summary_overall.csv
- /content/cs6493_topic4_generation/outputs/qwen_7b_topic4_generation_raw_results.jsonl


## Quick qualitative comparison

This cell shows a few side-by-side examples so you can inspect whether the instruction-tuned model is more stable, grounded, or concise.


In [21]:

wide_compare = (
    all_results[["id", "domain", "question", "answer", "model_label", "prediction"]]
    .pivot_table(
        index=["id", "domain", "question", "answer"],
        columns="model_label",
        values="prediction",
        aggfunc="first",
    )
    .reset_index()
)

display(wide_compare.head(10))


model_label,id,domain,question,answer,base_zero_shot,instruction_tuned
0,10808977,pubmedqa,Can tailored interventions increase mammography use among HMO women?,yes,Insufficient evidence.,Insufficient evidence.
1,10966337,pubmedqa,A short stay or 23-hour ward in a general and academic children's hospital: are they effective?,yes,Insufficient evidence.,Insufficient evidence.
2,16418930,pubmedqa,Landolt C and snellen e acuity: differences in strabismus amblyopia?,no,Insufficient evidence.,Insufficient evidence.
3,17113061,pubmedqa,Do mutations causing low HDL-C promote increased carotid intima-media thickness?,no,Insufficient evidence.,Insufficient evidence.
4,17208539,pubmedqa,Are the long-term results of the transanal pull-through equal to those of the transabdominal pull-through?,no,Insufficient evidence.,Insufficient evidence.
5,18847643,pubmedqa,Therapeutic anticoagulation in the trauma patient: is it safe?,no,Insufficient evidence.,Insufficient evidence.
6,21645374,pubmedqa,Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?,yes,Insufficient evidence.,Insufficient evidence.
7,23831910,pubmedqa,Double balloon enteroscopy: is it efficacious and safe in a community setting?,yes,Insufficient evidence.,Insufficient evidence.
8,25432938,pubmedqa,Did Chile's traffic law reform push police enforcement?,yes,Insufficient evidence.,Insufficient evidence.
9,26037986,pubmedqa,30-Day and 1-year mortality in emergency general surgery laparotomies: an area of concern and need for improvement?,maybe,Insufficient evidence.,Insufficient evidence.


## Optional: export a template JSONL file for your teammate

This is useful if you want to give your retrieval teammate a concrete schema to follow.


In [22]:

template_records = eval_df.head(2).to_dict("records")
template_path = OUTPUT_DIR / "retrieval_jsonl_template.jsonl"

with open(template_path, "w", encoding="utf-8") as f:
    for obj in template_records:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("Template saved to:", template_path)


Template saved to: /content/cs6493_topic4_generation/outputs/retrieval_jsonl_template.jsonl


## Optional: teammate repo reference

Your teammate's repo is:

`https://github.com/CS6493/RAGProjectPreprocess`

You do **not** need that repo for this notebook's **oracle mode**.

But if you want to keep everything in one Colab workspace, you can clone it here for reference:

```python
!git clone https://github.com/CS6493/RAGProjectPreprocess.git
```

For your final report, the best workflow is usually:

1. teammate runs retrieval and exports JSONL  
2. you switch this notebook to `INPUT_MODE = "team_jsonl"`  
3. you run the two generation models and compare outputs


## Troubleshooting

### 1) CUDA out of memory
Try one or more of these:
- switch `MODEL_PRESET` from `qwen_7b` to `qwen_3b`
- reduce `MAX_NEW_TOKENS`
- reduce `SAMPLES_PER_DOMAIN`
- restart runtime and run again
- keep `USE_4BIT = True`

### 2) Slow first run
The first run downloads models and datasets. Later runs are faster if you cache them in Google Drive.

### 3) Why oracle mode is not enough for the final report
Oracle mode is mainly for:
- debugging the generation module
- comparing base vs instruct behavior
- getting a self-contained Colab pipeline working quickly

But it does **not** properly test retrieval quality differences.  
For the real Topic 4 retrieval–generation analysis, use **team JSONL mode** with actual retrieval outputs.
